In [31]:
# NNSE Function-based Implementation for Tyson Model
# Implements a vector-based mutation and permutation algorithm

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import copy
import time

# ============================================================================
# === CONFIGURATION VARIABLES ===
# ============================================================================

# Simulation settings
N_STEPS = 100  # Number of mutation steps to run
SIGMA = 0.05  # Standard deviation for Gaussian mutations in normalized space
N_Vec = 50  # Number of bins for binning squared differences
MAX_VALUE = 50.0  # Maximum log value for bin thresholds (logspace goes from 0 to this)
K_INITIAL = int(N_Vec // 50)  # Number of top positions to fill initially (top half)
T_START = 0.0  # Simulation start time
T_END = 500.0  # Simulation end time
N_TIME_POINTS = 501  # Number of time points

# Define bin thresholds using logspace (equally spaced in log space)
# y0, y1, ..., yN_BINS: thresholds for binning
bin_thresholds = np.linspace(0, MAX_VALUE, N_Vec + 1) + MAX_VALUE / N_Vec
  # y0, y1, ..., yN_BINS, basically sets y0 to the value of y1 because function will likely never be zero

# ============================================================================
# === BASE PARAMETERS (p0) ===
# ============================================================================

p0 = {
    "k1_aa_over_CT": 0.015,
    "k2": 0.0,
    "k3_CT": 200.0,
    "k4": 180.0,
    "k4prime": 0.018,
    "k5_minusP": 0.0,
    "k6": 1.0,
    "k7": 0.6,
    "k8_minusP": 100.0,
    "k9": 50.0,
    "CT": 1.0
}

# Parameters to vary
param_names = [
    "k1_aa_over_CT",
    "k3_CT",
    "k4",
    "k4prime",
    "k6",
    "k7"
]

p0_vec = np.array([p0[name] for name in param_names])
n_params = len(param_names)

# Time evaluation array
t_eval = np.linspace(T_START, T_END, N_TIME_POINTS)

# ============================================================================
# === TYSON MODEL DEFINITION ===
# ============================================================================

CT = p0["CT"]

def F_M(M, p):
    """Helper function for M-dependent rate"""
    return p["k4prime"] + p["k4"] * (M / p["CT"])**2

def f_rhs(t, x, p):
    """Right-hand side of the ODE system"""
    # x = [C2, CP, pM, M, Y, YP]
    C2, CP, pM, M, Y, YP = x
    k3 = p["k3_CT"] / p["CT"]
    k1 = p["k1_aa_over_CT"] * p['CT']
    dC2 = p["k6"] * M - p["k8_minusP"] * C2 + p["k9"] * CP
    dCP = -k3 * CP * Y + p["k8_minusP"] * C2 - p["k9"] * CP
    dpM = k3 * CP * Y - pM * F_M(M, p) + p["k5_minusP"] * M
    dM  = pM * F_M(M, p) - p["k5_minusP"] * M - p["k6"] * M
    dY  = k1 - p["k2"] * Y - k3 * CP * Y
    dYP = p["k6"] * M - p["k7"] * YP
    return np.array([dC2, dCP, dpM, dM, dY, dYP])

def simulate_at_params(p_local, t_eval):
    """Simulate the ODE system with given parameters"""
    y0 = np.array([0.9, 0.05, 0.0, 0.005, 0.3, 0.0])
    sol = solve_ivp(lambda tt, xx: f_rhs(tt, xx, p_local), 
                    (t_eval[0], t_eval[-1]), y0,
                    method='BDF', t_eval=t_eval, rtol=1e-6, atol=1e-8)
    if not sol.success:
        raise RuntimeError("Integrator failed: " + sol.message)
    return sol.t, sol.y

def compute_obs(X):
    """Compute observables: YT/CT and M/CT"""
    if X.ndim == 1:
        C2, CP, pM, M, Y, YP = X
        YT = Y + YP + pM + M
        return YT / CT, M / CT
    else:
        C2, CP, pM, M, Y, YP = X
        YT = Y + YP + pM + M
        return YT / CT, M / CT

# ============================================================================
# === REFERENCE SIMULATION (p0) ===
# ============================================================================

print("Running reference simulation with p0...")
t0, y0 = simulate_at_params(p0, t_eval)
YT0, M0 = compute_obs(y0)
print(f"✓ Reference simulation complete")

# ============================================================================
# === SIM FUNCTION ===
# ============================================================================

def sim(P_vec):
    """
    Simulate with parameter vector P and compute squared difference with p0.
    Returns the squared difference (f(xi) value).
    """
    # Convert parameter vector to dictionary
    p_local = copy.deepcopy(p0)
    for i, name in enumerate(param_names):
        p_local[name] = P_vec[i]
    
    # Run simulation
    try:
        t, y = simulate_at_params(p_local, t_eval)
        YT, M = compute_obs(y)
        
        # Interpolate reference to match time points
        YT0_interp = np.interp(t, t0, YT0)
        M0_interp = np.interp(t, t0, M0)
        
        # Compute squared differences
        diff_YT_sq = (YT - YT0_interp)**2
        diff_M_sq = (M - M0_interp)**2
        
        # Integrate squared differences
        integral_YT_sq = np.trapz(diff_YT_sq, t)
        integral_M_sq = np.trapz(diff_M_sq, t)
        squared_diff = integral_YT_sq + integral_M_sq
        
        return squared_diff
        
    except Exception as e:
        # If simulation fails, return infinity
        print(f"Warning: Simulation failed: {e}")
        return np.inf

print(f"✓ Configuration complete")
print(f"  Parameters: {param_names}")
print(f"  Number of parameters: {n_params}")
print(f"  Bin thresholds (y0, ..., y{N_Vec}): [{bin_thresholds[0]:.4e}, ..., {bin_thresholds[-1]:.4e}]")


Running reference simulation with p0...
✓ Reference simulation complete
✓ Configuration complete
  Parameters: ['k1_aa_over_CT', 'k3_CT', 'k4', 'k4prime', 'k6', 'k7']
  Number of parameters: 6
  Bin thresholds (y0, ..., y50): [1.0000e+00, ..., 5.1000e+01]


In [32]:
# ============================================================================
# === TYSONFUNC: MUTATION AND PERMUTATION FUNCTION ===
# ============================================================================

def TysonFunc(X_list, fX_list):
    """
    Mutate each parameter vector, evaluate, reject if worse, then permute.
    Also handles filling empty positions: if position 0 is empty after swaps,
    generates a new point and tries to place it appropriately.
    
    Args:
        X_list: List of parameter vectors [x0, x1, ..., xn] where each xi is a numpy array (may be None)
        fX_list: List of function values [f(x0), f(x1), ..., f(xn)] (may be None)
    
    Returns:
        v_list: List of parameter vectors after mutation and permutation [v0, v1, ..., vn]
        fv_list: List of function values [f(v0), f(v1), ..., f(vn)]
        fX_prime_list: List of function values after mutation but before permutation [f(x'0), f(x'1), ..., f(x'n)]
        swaps: List of (i, j) tuples indicating which positions were swapped
    """
    n = len(X_list)
    
    # Step 1: Mutate each non-empty xi
    X_prime_list = []
    fX_prime_list = []
    
    for i in range(n):
        xi = X_list[i]
        fxi = fX_list[i]
        
        # Skip empty positions
        if xi is None or fxi is None:
            X_prime_list.append(None)
            fX_prime_list.append(None)
            continue
        
        # Normalize parameters: u_i = p_i / (2 * p0_i) so each lies in [0,1]
        u_vec = xi / (2.0 * p0_vec)
        
        # Apply Gaussian mutation in u-space
        u_mutated = u_vec + np.random.normal(0, SIGMA, size=n_params)
        
        # Wrap around boundaries [0, 1] with periodic boundary conditions
        u_mutated = u_mutated % 1.0
        
        # Map back to parameter space: p_i = 2 * p0_i * u_i
        xi_prime = 2.0 * p0_vec * u_mutated
        
        # Evaluate mutated parameter: f(x'i) := sim(x'i)
        fxi_prime = sim(xi_prime)
        
        # Step 2: Reject x'i if f(x'i) > y_i, otherwise accept
        if fxi_prime > bin_thresholds[i]:
            # Reject: keep original
            X_prime_list.append(xi.copy())
            fX_prime_list.append(fxi)
        else:
            # Accept: use mutated
            X_prime_list.append(xi_prime)
            fX_prime_list.append(fxi_prime)
    
    # Step 3: Permutation step
    # For each position i from n-1 down to 1 (0-indexed: n-1, n-2, ..., 1)
    # If f(x'_i) <= y_{i-1}, swap position i with position i-1
    v_list = X_prime_list.copy()
    fv_list = fX_prime_list.copy()
    
    # Track swaps
    swaps = []  # List of (i, j) tuples for swaps
    swapped_with_empty = False  # Track if any swap involved an empty position
    
    for i in range(n-1, 0, -1):  # i from n-1 down to 1
        # Skip if both positions are empty
        if fv_list[i] is None and fv_list[i-1] is None:
            continue
        
        # If f(x'_i) <= y_{i-1} and position i is not empty, swap
        if fv_list[i] is not None and fv_list[i] <= bin_thresholds[i-1]:
            # Check if swapping with empty
            if fv_list[i-1] is None:
                swapped_with_empty = True
            
            # Swap position i with position i-1
            if v_list[i] is not None and v_list[i-1] is not None:
                v_list[i], v_list[i-1] = v_list[i-1].copy(), v_list[i].copy()
            else:
                v_list[i], v_list[i-1] = v_list[i-1], v_list[i]
            fv_list[i], fv_list[i-1] = fv_list[i-1], fv_list[i]
            swaps.append((i, i-1))  # Record the swap
    
    # Step 4: Fill position 0 if it's empty (after swaps)
    # If position 0 is empty, generate a new point and try to place it
    if v_list[0] is None or fv_list[0] is None:
        # Generate a new random point
        max_attempts = 1000
        for attempt in range(max_attempts):
            # Generate random point in normalized u-space [0, 1]
            u_random = np.random.uniform(0, 1, size=n_params)
            # Map to parameter space: p_i = 2 * p0_i * u_i
            x0_new = 2.0 * p0_vec * u_random
            fx0_new = sim(x0_new)
            
            # Check if it can be kept at position 0
            if fx0_new <= bin_thresholds[0]:
                # Keep it at position 0
                v_list[0] = x0_new
                fv_list[0] = fx0_new
                break
            else:
                # Try to swap it down to a position where it fits
                placed = False
                for j in range(1, n):
                    if fx0_new <= bin_thresholds[j]:
                        # Can place it at position j
                        # If position j is empty, just place it there
                        if v_list[j] is None or fv_list[j] is None:
                            v_list[j] = x0_new
                            fv_list[j] = fx0_new
                            placed = True
                            break
                        # If position j has a worse value, swap
                        elif fv_list[j] is not None and fx0_new < fv_list[j]:
                            v_list[j] = x0_new
                            fv_list[j] = fx0_new
                            placed = True
                            break
                
                if placed:
                    break
                # If we couldn't place it anywhere, try again with a new random point
    
    return v_list, fv_list, fX_prime_list, swaps

print("✓ TysonFunc defined")


✓ TysonFunc defined


In [33]:
# ============================================================================
# === SIMULATION LOOP ===
# ============================================================================
# Note: fill_vacancies logic is now integrated into TysonFunc

# Helper function to format f values and y thresholds, handling None
def format_f_vals(fX_list, n):
    """Format f values and y thresholds for printing, handling None values.
    Returns two strings: (f_vals_str, y_vals_str)
    Always shows all f(xi) and y_i values.
    """
    indices = list(range(n))
    f_vals = [fX_list[i] if i < len(fX_list) else None for i in indices]
    f_vals_str = " ".join([f"{fx:<12.3e}" if fx is not None else f"{'---':<12}" for fx in f_vals])
    y_vals_str = " ".join([f"{bin_thresholds[i]:<12.3e}" for i in indices])
    return f_vals_str, y_vals_str

n = N_Vec  # Number of parameter vectors to maintain

print(f"Starting simulation with n={n} parameter vectors for {N_STEPS} steps...")

# Initialize: Only fill worst K positions (highest indices), leave best positions empty (None)
K = K_INITIAL
X_list = [None] * n
fX_list = [None] * n

print(f"Generating initial random parameter vectors (filling worst {K} positions, indices {n-K} to {n-1})...")

# Generate K random points in the worst positions (highest indices, highest thresholds)
# Fill positions from n-K to n-1 (e.g., if n=50, K=25, fill positions 25-49)
for idx in range(n-K, n):
    # Generate random point in normalized u-space [0, 1]
    u_random = np.random.uniform(0, 1, size=n_params)
    # Map to parameter space: p_i = 2 * p0_i * u_i
    xi = 2.0 * p0_vec * u_random
    fxi = sim(xi)
    X_list[idx] = xi
    fX_list[idx] = fxi

# Don't sort during initialization - keep filled positions in their original positions (worst positions)
# The sorting will happen naturally through the mutation and swapping process
filled_count = sum(1 for x in X_list if x is not None)
print(f"\nFilled: {filled_count} positions")



# Print after sorting (so position 0 shows the best value)
filled_count = len(filled_pairs_sorted)
print(f"\nFilled: {filled_count} positions")

# Use the same format as the main table
if n <= 7:
    header_cols = [f"f(x{i})" for i in range(filled_count)]
    header = f"{'Index':<8} " + " ".join([f"{col:<12}" for col in header_cols])
    y_header_cols = [f"y{i}" for i in range(filled_count)]
    y_header = f"{'':<8} " + " ".join([f"{col:<12}" for col in y_header_cols])
    separator_len = 8 + 1 + filled_count * 13
else:
    header_cols = [f"f(x{i})" for i in range(5)] + [f"f(x{filled_count//2})", f"f(x{filled_count-1})"]
    header = f"{'Index':<8} " + " ".join([f"{col:<12}" for col in header_cols])
    y_header_cols = [f"y{i}" for i in range(5)] + [f"y{filled_count//2}", f"y{filled_count-1}"]
    y_header = f"{'':<8} " + " ".join([f"{col:<12}" for col in y_header_cols])
    separator_len = 8 + 1 + 7 * 13

print(header)
print(y_header)
print("-" * separator_len)

# Print the values using format_f_vals
f_vals_str, y_vals_str = format_f_vals(fX_list, n)
print(f"{'Init':<8} {f_vals_str}")
print(f"{'':<8} {y_vals_str}")

# Show empty positions info
if filled_count < n:
    print(f"\nPositions {filled_count} to {n-1} are empty (best positions, will be filled during simulation)")

print(f"\n✓ Initialization complete ({filled_count} positions filled, {n-filled_count} empty)")

# Storage for tracking
all_X = [[x.copy() if x is not None else None for x in X_list]]
all_fX = [[fx if fx is not None else None for fx in fX_list]]
all_swaps = [[]]  # Store swaps for each step (empty for initial state)

# Progress tracking
print_interval = 1  # Print every 5%

# Create table header
# Reordered: Step, #swap, #empty, then all f(xi) values
header_cols = [f"f(x{i})" for i in range(n)]
header = f"{'Step':<6} {'#swap':<6} {'#empty':<7} " + " ".join([f"{col:<12}" for col in header_cols])
separator_len = 6 + 1 + 6 + 1 + 7 + 1 + n * 13  # Step + #swap + #empty + n columns
print(f"\n{header}")
# Print y thresholds row
y_header_cols = [f"y{i}" for i in range(n)]
y_header = f"{'':<6} {'':<6} {'':<7} " + " ".join([f"{col:<12}" for col in y_header_cols])
print(y_header)
print("-" * separator_len)

# Print initial state
f_vals_str, y_vals_str = format_f_vals(fX_list, n)
empty_count = sum(1 for i in range(len(X_list)) if X_list[i] is None or fX_list[i] is None)
print(f"{'Init':<6} {'0':<6} {empty_count:<7} {f_vals_str}")
print(f"{'':<6} {'':<6} {'':<7} {y_vals_str}")

# Main simulation loop
for step in range(N_STEPS):
    # Store unmutated state (before mutation)
    fX_unmutated = [fx if fx is not None else None for fx in fX_list]
    
    # Apply TysonFunc (now includes fill spots logic)
    X_list, fX_list, fX_mutated, swaps = TysonFunc(X_list, fX_list)
    
    # Store trajectory (handle None values)
    all_X.append([x.copy() if x is not None else None for x in X_list])
    all_fX.append([fx if fx is not None else None for fx in fX_list])
    all_swaps.append(swaps)
    
    # Progress report in table format
    if (step + 1) % print_interval == 0 or step == 0:
        # Count empty positions in unmutated state
        empty_count_unmut = sum(1 for i in range(len(X_list)) if X_list[i] is None or fX_unmutated[i] is None)
        
        # Print unmutated row
        f_vals_str, y_vals_str = format_f_vals(fX_unmutated, n)
        print(f"{step+1:<6} {'-':<6} {empty_count_unmut:<7} {f_vals_str}")
        print(f"{'':<6} {'':<6} {'':<7} {y_vals_str}")
        
        # Count empty positions in current state
        empty_count = sum(1 for i in range(len(X_list)) if X_list[i] is None or fX_list[i] is None)
        
        # Print mutated row
        f_vals_str, y_vals_str = format_f_vals(fX_mutated, n)
        print(f"{'':<6} {len(swaps):<6} {empty_count:<7} {f_vals_str}")
        print(f"{'':<6} {'':<6} {'':<7} {y_vals_str}")

print(f"\n✓ Simulation complete!")
# Print final state in table format
f_vals_str, y_vals_str = format_f_vals(fX_list, n)
empty_count = sum(1 for i in range(len(X_list)) if X_list[i] is None or fX_list[i] is None)
print(f"{'Final':<6} {'0':<6} {empty_count:<7} {f_vals_str}")
print(f"{'':<6} {'':<6} {'':<7} {y_vals_str}")

# Convert to numpy arrays for easier analysis
all_X_array = np.array(all_X)  # Shape: (N_STEPS+1, n, n_params)
all_fX_array = np.array(all_fX)  # Shape: (N_STEPS+1, n)


Starting simulation with n=50 parameter vectors for 100 steps...
Generating initial random parameter vectors (filling worst 1 positions, indices 49 to 49)...

Filled: 1 positions

Filled: 1 positions
Index    f(x0)        f(x1)        f(x2)        f(x3)        f(x4)        f(x0)        f(x0)       
         y0           y1           y2           y3           y4           y0           y0          
----------------------------------------------------------------------------------------------------
Init     ---          ---          ---          ---          ---          ---          ---          ---          ---          ---          ---          ---          ---          ---          ---          ---          ---          ---          ---          ---          ---          ---          ---          ---          ---          ---          ---          ---          ---          ---          ---          ---          ---          ---          ---          ---          ---          ---      

ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 2 dimensions. The detected shape was (101, 50) + inhomogeneous part.

In [ ]:
# ============================================================================
# === VISUALIZATION: PCA TRAJECTORY AND DISTRIBUTIONS ===
# ============================================================================

from sklearn.decomposition import PCA
from scipy.stats import gaussian_kde

print("Preparing data for visualization...")

# Convert to numpy arrays, handling None values
# For X: use np.nan where None, for fX: use np.nan where None
all_X_array = np.array([[x if x is not None else np.nan * np.ones(n_params) for x in step_X] for step_X in all_X])
all_fX_array = np.array([[fx if fx is not None else np.nan for fx in step_fX] for step_fX in all_fX])

# Extract the best vector (lowest f value) at each step
# Since vectors are sorted by f value, index 0 is always the best (if not None)
# all_X_array shape: (N_STEPS+1, n, n_params) -> extract best: (N_STEPS+1, n_params)
trajectory = all_X_array[:, 0, :]  # Best parameter vector at each step
squared_diffs = all_fX_array[:, 0]  # Best f value at each step

# Filter out steps where best vector is None (shouldn't happen, but just in case)
valid_mask = ~np.isnan(squared_diffs)
trajectory = trajectory[valid_mask]
squared_diffs = squared_diffs[valid_mask]

print(f"  Total steps: {len(trajectory)}")
print(f"  Best f value trajectory: {len(squared_diffs)} points")

# ============================================================================
# === PCA TRAJECTORY PLOT ===
# ============================================================================

print("\nComputing PCA for trajectory visualization...")

# Normalize trajectory for PCA (use standardized parameters)
trajectory_normalized = (trajectory - trajectory.mean(axis=0)) / (trajectory.std(axis=0) + 1e-10)

# Compute PCA
pca = PCA(n_components=min(3, n_params))
trajectory_pca = pca.fit_transform(trajectory_normalized)

print(f"✓ PCA complete")
print(f"  Explained variance ratio: {pca.explained_variance_ratio_}")
print(f"  Total explained variance: {np.sum(pca.explained_variance_ratio_):.4f}")

# Simulate with first and last parameters for YT/CT comparison
print("Simulating with first and last parameters...")
P_first = trajectory[0]  # Best parameter vector at step 0
P_last = trajectory[-1]  # Best parameter vector at final step

# Convert to parameter dictionaries
p_first = copy.deepcopy(p0)
p_last = copy.deepcopy(p0)
for i, name in enumerate(param_names):
    p_first[name] = P_first[i]
    p_last[name] = P_last[i]

# Simulate
t_first, y_first = simulate_at_params(p_first, t_eval)
YT_first, M_first = compute_obs(y_first)
t_last, y_last = simulate_at_params(p_last, t_eval)
YT_last, M_last = compute_obs(y_last)

print("✓ Simulations complete")

# Create figure with 3 subplots
fig = plt.figure(figsize=(20, 6))
gs = fig.add_gridspec(1, 3, hspace=0.3, wspace=0.3)

# Plot 2D PCA trajectory
ax1 = fig.add_subplot(gs[0, 0])
scatter = ax1.scatter(trajectory_pca[:, 0], trajectory_pca[:, 1], 
                     c=range(len(trajectory_pca)), cmap='viridis', 
                     s=20, alpha=0.6, edgecolors='none')
ax1.plot(trajectory_pca[:, 0], trajectory_pca[:, 1], 'k-', alpha=0.3, linewidth=0.5)
ax1.scatter(trajectory_pca[0, 0], trajectory_pca[0, 1], 
           color='red', s=100, marker='o', label='Start', zorder=5)
ax1.scatter(trajectory_pca[-1, 0], trajectory_pca[-1, 1], 
           color='blue', s=100, marker='s', label='End', zorder=5)
ax1.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)', fontsize=12)
ax1.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)', fontsize=12)
ax1.set_title('NNSE Trajectory in PCA Space (PC1 vs PC2)', fontsize=14, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)
plt.colorbar(scatter, ax=ax1, label='Step')

# Second subplot: 3D plot or squared difference over time
if trajectory_pca.shape[1] >= 3:
    from mpl_toolkits.mplot3d import Axes3D
    ax2 = fig.add_subplot(gs[0, 1], projection='3d')
    scatter2 = ax2.scatter(trajectory_pca[:, 0], trajectory_pca[:, 1], trajectory_pca[:, 2],
                         c=range(len(trajectory_pca)), cmap='viridis', s=20, alpha=0.6)
    ax2.plot(trajectory_pca[:, 0], trajectory_pca[:, 1], trajectory_pca[:, 2], 
            'k-', alpha=0.3, linewidth=0.5)
    ax2.scatter(trajectory_pca[0, 0], trajectory_pca[0, 1], trajectory_pca[0, 2],
               color='red', s=100, marker='o', label='Start')
    ax2.scatter(trajectory_pca[-1, 0], trajectory_pca[-1, 1], trajectory_pca[-1, 2],
               color='blue', s=100, marker='s', label='End')
    ax2.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%})', fontsize=10)
    ax2.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%})', fontsize=10)
    ax2.set_zlabel(f'PC3 ({pca.explained_variance_ratio_[2]:.2%})', fontsize=10)
    ax2.set_title('NNSE Trajectory in PCA Space (3D)', fontsize=14, fontweight='bold')
    ax2.legend(fontsize=10)
    plt.colorbar(scatter2, ax=ax2, label='Step')
else:
    # If only 2 components, show squared difference over time
    ax2 = fig.add_subplot(gs[0, 1])
    ax2.plot(range(len(squared_diffs)), squared_diffs, 'b-', linewidth=1, alpha=0.7)
    ax2.set_xlabel('Step', fontsize=12)
    ax2.set_ylabel('Squared Difference', fontsize=12)
    ax2.set_title('Best Squared Difference Over Time', fontsize=14, fontweight='bold')
    ax2.set_yscale('log')
    ax2.grid(True, alpha=0.3)

# Third subplot: YT/CT comparison
ax3 = fig.add_subplot(gs[0, 2])
ax3.plot(t0, YT0, 'k-', lw=2, label='Reference (p0)', alpha=0.8)
ax3.plot(t_first, YT_first, 'r-', lw=2, label='First parameter', alpha=0.7)
ax3.plot(t_last, YT_last, 'b-', lw=2, label='Last parameter', alpha=0.7)
ax3.set_xlabel('Time (min)', fontsize=12)
ax3.set_ylabel('YT/CT', fontsize=12)
ax3.set_title('YT/CT Comparison', fontsize=14, fontweight='bold')
ax3.legend(fontsize=10)
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# ============================================================================
# === DISTRIBUTION OF SQUARED DIFFERENCES ===
# ============================================================================

print("\nPlotting distribution of squared differences...")

# For distribution, use all squared differences from all vectors at all steps
# This gives a better picture of the full distribution
squared_diffs_all = all_fX_array.flatten()  # All function values from all vectors

# Filter out infinite values
valid_mask = np.isfinite(squared_diffs_all)
squared_diffs_valid = squared_diffs_all[valid_mask]

print(f"  Valid values: {np.sum(valid_mask)}/{len(squared_diffs_all)}")
print(f"  Mean: {np.mean(squared_diffs_valid):.6e}")
print(f"  Median: {np.median(squared_diffs_valid):.6e}")
print(f"  Min: {np.min(squared_diffs_valid):.6e}")
print(f"  Max: {np.max(squared_diffs_valid):.6e}")

# Create distribution plot
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Histogram with linear bins
ax1 = axes[0]
n_bins_hist = 50
# Use linear bins from min to max
counts, bins_hist, patches = ax1.hist(squared_diffs_valid, bins=n_bins_hist, 
                                      edgecolor='black', alpha=0.7, color='steelblue',
                                      density=True)
ax1.axvline(np.mean(squared_diffs_valid), color='red', linestyle='--', 
           linewidth=2, label=f'Mean: {np.mean(squared_diffs_valid):.4e}')
ax1.axvline(np.median(squared_diffs_valid), color='green', linestyle='--', 
           linewidth=2, label=f'Median: {np.median(squared_diffs_valid):.4e}')
ax1.set_xlabel('Squared Difference', fontsize=12, fontweight='bold')
ax1.set_ylabel('Density', fontsize=12, fontweight='bold')
ax1.set_title('Distribution of Squared Differences (Histogram)', 
             fontsize=14, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Kernel density estimate (KDE) with linear scale
ax2 = axes[1]
if len(squared_diffs_valid) > 1:
    # Use linear KDE (not log-transformed)
    kde = gaussian_kde(squared_diffs_valid)
    x_kde = np.linspace(squared_diffs_valid.min(), squared_diffs_valid.max(), 200)
    density = kde(x_kde)
    ax2.plot(x_kde, density, 'b-', linewidth=2, label='KDE')
    ax2.fill_between(x_kde, 0, density, alpha=0.3, color='steelblue')
    ax2.axvline(np.mean(squared_diffs_valid), color='red', linestyle='--', 
               linewidth=2, label=f'Mean: {np.mean(squared_diffs_valid):.4e}')
    ax2.axvline(np.median(squared_diffs_valid), color='green', linestyle='--', 
               linewidth=2, label=f'Median: {np.median(squared_diffs_valid):.4e}')
    ax2.set_xlabel('Squared Difference', fontsize=12, fontweight='bold')
    ax2.set_ylabel('Density', fontsize=12, fontweight='bold')
    ax2.set_title('Distribution of Squared Differences (KDE)', 
                 fontsize=14, fontweight='bold')
    ax2.legend(fontsize=10)
    ax2.grid(True, alpha=0.3)
else:
    ax2.text(0.5, 0.5, 'Not enough data for KDE', 
            ha='center', va='center', transform=ax2.transAxes, fontsize=12)
    ax2.set_title('Distribution of Squared Differences (KDE)', 
                 fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print("✓ Visualization complete")
